# 6. Comment-gap score

Port the corrected legacy comment-gap statistic to the shared 2025 model sample. Within each discussion, audience `relative_votes` are ranked descending with equal midranks for ties. If there are `N` candidates and `k` curator picks, the score is `(mean curator-pick rank - (k + 1) / 2) / (N - k)`: 0 is the best possible k-set, 1 is the worst, and a random k-set has expectation 0.5.

The primary scope is `all`; use `COMMENTGAP_MODEL_SCOPES=all,root` for the root-only appendix sensitivity. Stage 5 must run first because it freezes and audits the article-topic join.

In [ ]:
from pathlib import Path
import os

from commentgap_analysis.comment_gap import run_comment_gap_analysis
from commentgap_analysis.category_labels import translate_news_category

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all, root")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")

MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
DESCRIPTIVES_ROOT = Path(os.getenv("COMMENTGAP_DESCRIPTIVES_ROOT", "model_output/selection_2025/paper1/descriptives"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_COMMENT_GAP_ROOT", "model_output/selection_2025/paper1/comment_gap"))
THREADS = int(os.getenv("COMMENTGAP_GAP_THREADS", "4"))
{"scopes": SCOPES, "descriptives": str(DESCRIPTIVES_ROOT), "output": str(OUTPUT_ROOT)}

In [ ]:
manifest = run_comment_gap_analysis(
    model_data_root=MODEL_DATA_ROOT,
    descriptives_root=DESCRIPTIVES_ROOT,
    output_root=OUTPUT_ROOT,
    scopes=SCOPES,
    threads=THREADS,
    make_figures=True,
)
manifest

## Key results

Lower comment-gap scores indicate closer curator–audience agreement; 0.5 is the random-set expectation. The ordinary mean and median give every discussion equal weight. Comment-weighted summaries weight each discussion by its number of candidate comments. The weighted median is the first ordered discussion gap where cumulative candidate-comment weight reaches 50%. The summaries below show these estimands, exact top-k overlap, a secondary legacy-style Jaccard diagnostic, and topic variation using the canonical stage outputs.

The Jaccard measure is imperfect: it discards the rank within each selected set, depends on the number of curator picks and the tie policy, and can be affected by exposure feedback because pinned comments may receive more votes. We therefore report it only as a secondary descriptive comparison, not as a definitive or causal measure of preference disagreement.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from commentgap_analysis.comment_gap import (
    plot_comment_gap_by_topic, plot_comment_gap_distribution,
)

gap_summary = pd.read_csv(OUTPUT_ROOT / "comment_gap_summary.csv")
gap_topics = pd.read_csv(OUTPUT_ROOT / "comment_gap_topic_summary.csv")
article_scores = pd.read_parquet(OUTPUT_ROOT / "article_gap_scores.parquet")

# Legacy comparison: J(A, B) = |A ∩ B| / |A ∪ B|. Here A is the
# curator-pick set and B is each audience top-k tie draw, with k equal
# to the number of curator picks in that discussion. Since overlap is
# stored as |A ∩ B|/k, J(A, B) = overlap / (2 - overlap).
overlap_columns = sorted(c for c in article_scores if c.startswith("curator_audience_overlap_draw_"))
stored_jaccard_columns = sorted(c for c in article_scores if c.startswith("curator_audience_jaccard_draw_"))
if not overlap_columns or len(overlap_columns) != len(stored_jaccard_columns):
    raise ValueError("Expected one overlap and Jaccard value for each audience tie draw")
overlap_values = article_scores[overlap_columns].to_numpy(dtype=float)
n_picks = article_scores["n_picks"].to_numpy(dtype=float)[:, None]
intersection_values = overlap_values * n_picks
jaccard_values = intersection_values / (2 * n_picks - intersection_values)
np.testing.assert_allclose(
    jaccard_values, article_scores[stored_jaccard_columns].to_numpy(dtype=float),
    equal_nan=True,
)
article_scores["jaccard_mean"] = jaccard_values.mean(axis=1)
article_scores["jaccard_gap_mean"] = 1 - article_scores["jaccard_mean"]

def jaccard_summary(frame, groups):
    return frame.groupby(groups, as_index=False, observed=True)[["jaccard_mean", "jaccard_gap_mean"]].mean()

jaccard_summary_all = pd.concat(
    [
        article_scores,
        article_scores.assign(analysis_partition="all_partitions"),
    ],
    ignore_index=True,
)
gap_summary = gap_summary.merge(
    jaccard_summary(jaccard_summary_all, ["scope", "analysis_partition"]),
    on=["scope", "analysis_partition"], validate="one_to_one",
)
gap_topics = gap_topics.merge(
    jaccard_summary(jaccard_summary_all, ["scope", "analysis_partition", "primary_topic"]),
    on=["scope", "analysis_partition", "primary_topic"], validate="one_to_one",
)
if "primary_topic_label" not in gap_topics:
    gap_topics["primary_topic_label"] = gap_topics["primary_topic"].map(translate_news_category)
primary_scope = "all" if "all" in SCOPES else SCOPES[0]
overall = gap_summary.query("analysis_partition == 'all_partitions'").copy()
overall_display = overall[[
    "scope", "n_articles", "gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median", "gap_q25", "gap_q75",
    "overlap_mean", "jaccard_mean", "jaccard_gap_mean", "candidates_mean", "picks_mean", "articles_with_vote_ties",
]].rename(columns={
    "gap_mean": "article-weighted mean gap",
    "gap_comment_weighted_mean": "comment-weighted mean gap",
    "gap_median": "median gap",
    "gap_comment_weighted_median": "comment-weighted median gap",
    "gap_q25": "gap Q1", "gap_q75": "gap Q3",
    "overlap_mean": "mean top-k overlap",
    "jaccard_mean": "mean Jaccard (secondary)",
    "jaccard_gap_mean": "mean Jaccard gap (secondary)",
})
display(
    overall_display.style.format({
        "n_articles": "{:,.0f}", "article-weighted mean gap": "{:.3f}",
        "comment-weighted mean gap": "{:.3f}", "median gap": "{:.3f}",
        "comment-weighted median gap": "{:.3f}",
        "gap Q1": "{:.3f}", "gap Q3": "{:.3f}", "mean top-k overlap": "{:.1%}",
        "mean Jaccard (secondary)": "{:.1%}",
        "mean Jaccard gap (secondary)": "{:.1%}",
        "candidates_mean": "{:,.1f}", "picks_mean": "{:.2f}",
        "articles_with_vote_ties": "{:,.0f}",
    }).hide(axis="index").set_caption("Overall comment-gap results")
)
display(plot_comment_gap_distribution(
    pd.read_parquet(OUTPUT_ROOT / "article_gap_scores.parquet"),
    primary_scope, output_root=None, show=False,
))

In [ ]:
primary_topics = gap_topics.query(
    "scope == @primary_scope and analysis_partition == 'all_partitions'"
).copy()
largest_topics = (
    primary_topics.nlargest(15, "n_articles")
    .sort_values("gap_mean")
    [["primary_topic_label", "n_articles", "gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median", "overlap_mean", "jaccard_mean", "jaccard_gap_mean"]]
)
largest_topics = largest_topics.rename(columns={"primary_topic_label": "primary_topic"})
display(
    largest_topics.style.format({
        "n_articles": "{:,.0f}", "gap_mean": "{:.3f}",
        "gap_comment_weighted_mean": "{:.3f}",
        "gap_median": "{:.3f}", "gap_comment_weighted_median": "{:.3f}",
        "overlap_mean": "{:.1%}", "jaccard_mean": "{:.1%}", "jaccard_gap_mean": "{:.1%}",
    }).background_gradient(subset=["gap_mean", "gap_comment_weighted_mean", "gap_median", "gap_comment_weighted_median"], cmap="RdYlGn_r", vmin=0, vmax=0.5)
    .hide(axis="index").set_caption("Comment gap in the 15 largest topics")
)
display(plot_comment_gap_by_topic(gap_topics, primary_scope, output_root=None, show=False))

## Output contract

The stage writes an article-level Parquet table, overall and topic CSV summaries, ten-draw curator/audience overlap and Jaccard diagnostics, PNG/PDF figures, and a hash manifest. The Jaccard value and its complement (the Jaccard gap) are secondary legacy-style set comparisons; the normalized rank gap remains the primary measure. The obsolete beta regressions in the legacy Rmd are not reproduced: their pre-2025 covariates are not equivalent to the 2025 feature contract, while stage 7 is the paper's inferential selection analysis.